# arXiv

In [1]:
%reload_ext sql
%sql duckdb:///:memory:
%config SqlMagic.displaylimit = 0
%config SqlMagic.autopandas = False

The 'toml' package isn't installed. To load settings from pyproject.toml or ~/.jupysql/config, install with: pip install toml

Connecting to 'duckdb:///:memory:'

In [2]:
%%sql

CREATE VIEW arXiv_pdfs AS
SELECT *
FROM read_parquet('../data/output/arXiv_pdf_manifest.parquet');

CREATE VIEW arXiv_src AS
SELECT *
FROM read_parquet('../data/output/arXiv_src_manifest.parquet');

CREATE VIEW arxiv_extract AS
SELECT *
FROM read_parquet('../data/output/arxiv-extract/*.parquet');

CREATE VIEW arxiv_metadata AS
SELECT REPLACE(id, '/', '') AS arxiv_id, update_date, versions
FROM read_json_auto('../data/output/arxiv-metadata-oai-snapshot.json');


Running query in 'duckdb:///:memory:'

Count


In [3]:
%%sql

SELECT
  'src' AS Directory,
  format('{:,}', SUM(num_items)) AS num_files,
  format('{:,}', COUNT(*)) AS num_tars,
  format('{:,.2f}', SUM(size) / POWER(1024, 4)) AS tars_tib,
  format('${:,.2f}', SUM(size) / POWER(1024, 3) * 0.09) AS internet_egress_usd
FROM arXiv_src

UNION ALL

SELECT
  'pdf' AS Directory,
  format('{:,}', SUM(num_items)) AS num_files,
  format('{:,}', COUNT(*)) AS num_tars,
  format('{:,.2f}', SUM(size) / POWER(1024, 4)) AS tars_tib,
  format('${:,.2f}', SUM(size) / POWER(1024, 3) * 0.09) AS internet_egress_usd
FROM arXiv_pdfs;

Running query in 'duckdb:///:memory:'

Directory,num_files,num_tars,tars_tib,internet_egress_usd
src,"2,969,765","11,940",5.78,$533.11
pdf,"2,945,938","11,042",5.32,$490.70


In [4]:
%%sql

SELECT                                                                                                                                                                                                                                
    strftime('%Y', timestamp) AS year,                      
    COUNT(*) AS num_files,                                                                                                                                                                                                              
    ROUND(SUM(size) / 1e9, 2) AS total_gb,                                                                                                                                                                                              
    ROUND(AVG(size) / 1e6, 1) AS avg_size_mb,                                                                                                                                                                                           
    ROUND(MIN(size) / 1e6, 1) AS min_size_mb,                                                                                                                                                                                           
    ROUND(MAX(size) / 1e6, 1) AS max_size_mb
FROM arXiv_src                                                                                                                                                                                                                         
GROUP BY year                                             
ORDER BY year;

Running query in 'duckdb:///:memory:'

year,num_files,total_gb,avg_size_mb,min_size_mb,max_size_mb
2010,318,96.8,304.4,1.0,576.1
2011,25,11.25,450.0,1.8,658.7
2012,92,43.14,468.9,0.1,636.3
2013,107,54.58,510.1,7.8,586.9
2014,148,76.76,518.6,71.4,687.2
2015,128,65.67,513.0,10.5,675.3
2016,184,98.35,534.5,79.2,1910.6
2017,228,120.02,526.4,33.0,661.7
2018,293,154.09,525.9,28.9,674.9
2019,334,178.99,535.9,38.7,851.6


In [5]:
%%sql

select status, file_type, COUNT(*) AS count 
FROM arxiv_extract
GROUP BY status, file_type
ORDER BY COUNT DESC;

Running query in 'duckdb:///:memory:'

status,file_type,count
ok,tex,2629720
empty,pdf,237175
empty,tex,12289
empty,postscript,4314
empty,html,931
skipped,tex,197
timeout,tex,141
empty,unknown,1


In [6]:
%%sql

select status, file_type, COUNT(*) AS count 
FROM arxiv_extract
GROUP BY status, file_type
ORDER BY COUNT DESC;

Running query in 'duckdb:///:memory:'

status,file_type,count
ok,tex,2629720
empty,pdf,237175
empty,tex,12289
empty,postscript,4314
empty,html,931
skipped,tex,197
timeout,tex,141
empty,unknown,1


In [7]:
%%sql

SELECT
  CASE
    WHEN status = 'ok' THEN 'ok'
    ELSE 'not_ok'
  END AS status_group,
  COUNT(*) AS count
FROM arxiv_extract
GROUP BY 1
ORDER BY 1;

Running query in 'duckdb:///:memory:'

status_group,count
not_ok,255048
ok,2629720


In [8]:
%%sql
    
SELECT COUNT(DISTINCT arxiv_id) FROM arxiv_metadata;

Running query in 'duckdb:///:memory:'

count(DISTINCT arxiv_id)
3015119


In [9]:
%%sql

WITH a AS (
  SELECT DISTINCT arxiv_id
  FROM arxiv_extract
  WHERE arxiv_id IS NOT NULL
),
b AS (
  SELECT DISTINCT arxiv_id
  FROM arxiv_metadata
  WHERE arxiv_id IS NOT NULL
)
SELECT
  CASE
    WHEN a.arxiv_id IS NOT NULL AND b.arxiv_id IS NOT NULL THEN 'in_both'
    WHEN a.arxiv_id IS NOT NULL THEN 'only_in_arxiv_extract'
    ELSE 'only_in_arxiv_metadata'
  END AS membership,
  COUNT(*) AS count
FROM a
FULL OUTER JOIN b
  ON a.arxiv_id = b.arxiv_id
GROUP BY 1
ORDER BY 1;

Running query in 'duckdb:///:memory:'

membership,count
in_both,2884764
only_in_arxiv_extract,4
only_in_arxiv_metadata,130355


In [10]:
%%sql

WITH a AS (
  SELECT DISTINCT arxiv_id
  FROM arxiv_extract
  WHERE arxiv_id IS NOT NULL
),
b AS (
  SELECT DISTINCT arxiv_id
  FROM arxiv_metadata
  WHERE arxiv_id IS NOT NULL
)
SELECT a.arxiv_id
FROM a
LEFT JOIN b
  ON a.arxiv_id = b.arxiv_id
WHERE b.arxiv_id IS NULL
ORDER BY a.arxiv_id
LIMIT 100;

Running query in 'duckdb:///:memory:'

arxiv_id
2307.02646
2401.09755
2402.18611
acc-phys9607002


In [11]:
%%sql

WITH extract_status AS (
  SELECT
    arxiv_id,
    CASE
      WHEN status = 'ok' THEN 'ok'
      ELSE 'not_ok'
    END AS status_group
  FROM arxiv_extract
  WHERE arxiv_id IS NOT NULL
),
metadata_classified AS (
  SELECT
    m.arxiv_id,
    CASE
      WHEN e.arxiv_id IS NULL THEN 'not_in_arxiv_extract'
      WHEN e.status_group = 'not_ok' THEN 'not_ok_in_arxiv_extract'
      ELSE 'ok_in_arxiv_extract'
    END AS membership
  FROM arxiv_metadata m
  LEFT JOIN extract_status e
    ON m.arxiv_id = e.arxiv_id
  WHERE m.arxiv_id IS NOT NULL
)
SELECT
  membership,
  COUNT(*) AS count
FROM metadata_classified
WHERE membership IN ('not_ok_in_arxiv_extract', 'not_in_arxiv_extract')
GROUP BY 1

UNION ALL

SELECT
  'total_sum' AS membership,
  COUNT(*) AS count
FROM metadata_classified
WHERE membership IN ('not_ok_in_arxiv_extract', 'not_in_arxiv_extract');

Running query in 'duckdb:///:memory:'

membership,count
not_ok_in_arxiv_extract,255049
not_in_arxiv_extract,130371
total_sum,385420
